# Seats of two connected flights and their bid prices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/revenue_management/notebooks/revenue_management.ipynb)

An airline based in Riga flies Barcelona → Riga and Riga → Helsinki on the same day, with a connection in Riga. It sells three trips (BCN–RIX, RIX–HEL and BCN–HEL with the connection), each with three fares: nine products. A BCN–HEL ticket needs a seat on both flights. Before sales open, the revenue manager decides how many tickets of each product to put on sale. The notebook follows the decision in steps:

1. The flights and the products.
2. The rule «highest fare first».
3. A linear model of the two flights and its optimal plan, next to the plan of the rule.
4. The bid price of each flight and of each product.
5. An offer of a tour operator for connecting seats.
6. Another aircraft on one of the flights.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and drop-down lists): change the values and run the cell again. Outside Colab the form is a line such as `seats = 10`: edit the value and run the cell.

All the data and results are explained in the [document of the case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/revenue_management/revenue_management.md). The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the case from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/revenue_management"
    %pip install -q {url}
    # A session that already had the case keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "fares" or m.startswith("fares.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
import fares as fm

## 1. Flights and products

The first table gives the two flights and their capacity, in seats. The second gives the nine products: the trip, the fare, the flights the trip uses, the price of a one-way ticket (€/ticket), the forecast demand (tickets) and the seats one ticket uses on each of its flights. A Business ticket uses 1.5 seats, because Business uses the economy seats with the middle seat left free. The third table says what each fare includes.

If every forecast ticket were sold, how many seats would each flight need? Which products compete for the seats of Riga → Helsinki?

In [ ]:
# @title Data { display-mode: "form" }
fm.show_data()

## 2. The rule «highest fare first»

The rule takes the products from the highest price to the lowest, and sells each one up to its demand or until one of the flights it uses is full. It needs no model. The cell prints the order followed, the revenue of the plan and the seats used on each flight, and a table with the tickets of each product on sale out of its demand.

A BCN–HEL BASIC ticket, at 110 €, comes before a BCN–RIX BASIC ticket at 75 € and a RIX–HEL BASIC ticket at 45 €. What does it take from each flight?

In [ ]:
# @title Rule «highest fare first» { display-mode: "form" }
fm.rule();

## 3. Linear model and optimal plan

$L$ is the set of flights, with index $l$, and $P$ the set of products, with index $p$. $r_p$ is the price of a ticket (€/ticket), $D_p$ the forecast demand (tickets), $a_{pl}$ the seats one ticket of $p$ uses on flight $l$ (1.5 for Business, 1 for the economy fares, 0 if $p$ does not use $l$) and $C_l$ the capacity of flight $l$ (seats). The decision $x_p$ is the number of tickets of product $p$ put on sale, a continuous variable.

$$
\begin{aligned}
\max \quad & \sum_{p \in P} r_p\, x_p && && \text{(revenue)} \\
\text{s.t.} \quad & \sum_{p \in P} a_{pl}\, x_p \le C_l && \forall l \in L && \text{(capacity of each flight)} \\
& x_p \le D_p && \forall p \in P && \text{(demand of each product)} \\
& x_p \ge 0 && \forall p \in P
\end{aligned}
$$

A BCN–HEL ticket appears in the capacity constraints of both flights. The cell solves the model and compares its plan with the plan of the rule: the revenue of each, the tickets of each product and the difference, in tickets and in euros, also drawn as bars. The last table gives each constraint: what the plan uses, its right-hand side, whether it is **binding** (the plan uses all of it) and its **slack**.

Which products does the model sell more of than the rule, and which fewer? Why are 78 BCN–HEL BASIC tickets left unsold although each one pays more than a BCN–RIX BASIC ticket?

In [ ]:
# @title Optimal plan against the rule { display-mode: "form" }
fm.optimal_plan();

## 4. Bid prices

The **bid price** of a flight is the shadow price of its capacity: the rise in revenue with one more seat on that flight, in €/seat. It is the lowest revenue for which one more seat on that flight is worth selling. The first table gives the bid price of each flight. Choose a flight in the list: the cell solves the model again with one seat more and one seat less on it and prints the change in revenue.

The bid price of a product is the bid prices of the flights it uses times the seats it uses on each. The second table gives it for the nine products, next to the price, and the shadow price of the demand of each product: the rise in revenue with one more ticket of forecast demand.

No fare costs 35 €. Where does the bid price of Riga → Helsinki come from? Which products have a price equal to their bid price, and what does that say about their unsold demand?

In [ ]:
# @title Bid prices of the flights and of the products { display-mode: "form" }
flight = "RIX → HEL"  # @param ["BCN → RIX", "RIX → HEL"]
fm.bid_prices(flight)

## 5. Offer of a tour operator

A tour operator asks for BCN–HEL seats on top of the forecast demand. The bid prices estimate the change in revenue as the seats times the price offered minus the bid price of a BCN–HEL seat. The cell also solves the model again with the seats of the offer in the plan, and gives the products whose tickets change. The figure draws, for 0 up to the seats of the offer, the estimate (red dashed line) and the change found by solving again (black points).

The sliders set the number of seats and the price per seat. For 120 €/seat, up to how many seats does the estimate agree with solving again? What happens to the bid price of Riga → Helsinki after the offer is accepted?

In [ ]:
# @title Offer of BCN–HEL seats { display-mode: "form" }
seats = 10  # @param {type:"slider", min:1, max:40, step:1}
price = 120  # @param {type:"slider", min:60, max:200, step:5}
fm.tour_operator_offer(seats, price);

## 6. Another aircraft

The airline can fly one of the flights with another aircraft. Choose the flight, the seats of the other aircraft and how much more its operating cost per flight is than that of the current one (negative: a saving). The cell solves the model again and gives the products whose tickets change, the change in revenue, the estimate with the bid price of the flight, whether the new capacity is inside the range in which that bid price holds, and the net result: change in revenue minus change in cost.

For Riga → Helsinki, try 180 seats with 1,200 € more of cost, and 120 seats with 800 € less. In which of the two is the estimate with the bid price enough to decide?

In [ ]:
# @title Another aircraft { display-mode: "form" }
flight = "RIX → HEL"  # @param ["BCN → RIX", "RIX → HEL"]
seats = 180  # @param {type:"slider", min:60, max:300, step:5}
cost_change = 1200  # @param {type:"slider", min:-3000, max:3000, step:100}
fm.other_aircraft(flight, seats, cost_change);

All the numbers are explained in the [document of the case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/revenue_management/revenue_management.md), which also has a script that reproduces them.